In [1]:
import torch
from torch import nn
import os, random
import numpy as np

import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.tokenize import TreebankWordTokenizer
from datasets import load_dataset

def reset_random_seeds():
    os.environ['PYTHONHASHSEED']=str(2)
    torch.manual_seed(2)
    np.random.seed(2)
    random.seed(2)
    torch.cuda.manual_seed(2)
    torch.cuda.manual_seed_all(2)

In [2]:
import re
from string import punctuation
from tqdm.auto import tqdm
import contractions as con

def preprocess_text(x):

    x = str(x).replace('&amp;','and').replace('&quot;','').lower()
    x = re.sub(r'@[A-Za-z0-9_]{1,15}', 'USER', x)
    x = con.fix(x)
    x = re.sub(r'&#x[0-9A-Fa-f]+;','',x)
    x = re.sub(r'&#\d+;',"'",x)
    x = re.sub(r'[^\x00-\x7F]+', "'",x)
    
    url_pattern = r'http\S+|www\S+'
    x = re.sub(url_pattern, 'LINK', x)
    
    punct_to_keep = """!,.:#?"-;//%$(){}@^*+<=>\\|'"""
    punct = ''.join([p for p in punctuation if p not in punct_to_keep])
    trans = str.maketrans(punct, ' ' * len(punct))
    x = x.translate(trans)
    x = ''.join(x)
    x = re.sub(r'([!"#$%&\()*+,-./:;<=>?@\\^_`{|}~])\s*\1+', r'\1', x)
    x = re.sub(r'([!"#$%&\()*+,-./:;<=>?@\\^_`{|}~])', r' \1 ', x)
    x = re.sub(r'\s+', ' ', x).strip().replace("'s "," 's ")
    x = x.replace("\\'"," '").replace("'"," ' ")
    
    return re.sub(r'\s+', ' ', x).strip()

In [3]:
from collections import Counter

class WordTokenizer:

    def __init__(self, vocab_size=10000):

        self.vocab_size = vocab_size

        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        self.sos_token = "<sos>"
        self.eos_token = "<eos>"

        self.special_tokens = [
            self.pad_token,
            self.unk_token,
            self.sos_token,
            self.eos_token
        ]

        self.word2id = {}
        self.id2word = {}

    def fit(self, texts):

        counter = Counter()

        for text in texts:
            tokens = text.split()
            counter.update(tokens)

        most_common = counter.most_common(
            self.vocab_size - len(self.special_tokens)
        )

        vocab_words = [w for w, _ in most_common]

        vocab = self.special_tokens + vocab_words

        self.word2id = {w: i for i, w in enumerate(vocab)}
        self.id2word = {i: w for w, i in self.word2id.items()}

    def encode(self, text, add_special_tokens=True):

        tokens = text.split()

        ids = []

        if add_special_tokens:
            ids.append(self.word2id[self.sos_token])

        for token in tokens:
            ids.append(
                self.word2id.get(token, self.word2id[self.unk_token])
            )

        if add_special_tokens:
            ids.append(self.word2id[self.eos_token])

        return ids

    def decode(self, ids, remove_special=True):

        words = []

        for i in ids:
            w = self.id2word.get(i, self.unk_token)

            if remove_special and w in self.special_tokens:
                continue

            words.append(w)

        return " ".join(words)

    def pad(self, sequences, max_len=None):

        if max_len is None:
            max_len = max(len(s) for s in sequences)

        padded = []

        for seq in sequences:

            seq = seq[:max_len]

            padding = [self.word2id[self.pad_token]] * (max_len - len(seq))

            padded.append(seq + padding)

        return padded

    def encode_batch(self, texts, pad=True, max_len=None):

        sequences = [self.encode(t) for t in texts]

        if pad:
            sequences = self.pad(sequences, max_len)

        return sequences

In [4]:
class AttentionPool(nn.Module):
    
    def __init__(self,dim):
        super().__init__()
        self.dim = dim 
        self.score = None
        self.w = nn.Linear(dim,1,bias=False)
        
    def forward(self,x,mask=None):
        attn = self.w(x)
        if mask is not None:
            mask = mask[:,:,torch.newaxis]
            attn = torch.where(mask,-1e8,attn)
        attn = torch.softmax(attn,axis=1)
        self.score = attn
        x = x * attn
        return x.sum(1)

In [5]:
def load_glove_model(path, dim=50):
    glove_model = {}

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            parts = line.rstrip().split()
            
            if len(parts) < dim + 1:
                continue

            word = parts[0]
            vector = parts[-dim:] 

            try:
                embedding = np.array(vector, dtype=np.float32)
                glove_model[word] = embedding
            except ValueError:
                continue

    print(f"{len(glove_model)} words loaded!")
    return glove_model

In [6]:
file = "/media/bibek/New Volume/dolma_300_2024_1.2M.100_combined.txt"

emb = load_glove_model(file,dim=300)

1200001 words loaded!


In [7]:
def initialize_embeddings(vocab_size,embed_dim = 300):

    limit = np.sqrt(6 / (embed_dim + embed_dim))
    w_emb = np.random.uniform(-limit, limit, (vocab_size, embed_dim))

    for w,idx in tok.word2id.items():
        try: w_emb[idx] = emb[w]
        except: pass
        
    w_emb = torch.tensor(w_emb,dtype=torch.float32)
        
    return w_emb

In [8]:
from torchinfo import summary


class BaseClassifier(nn.Module):
    
    def __init__(self,vocab,glove_emb,dim=128,emb_dim=300):
        
        super().__init__()
        self.dim = dim
        self.vocab = vocab
        self.emb = nn.Embedding(vocab,emb_dim,_weight=glove_emb)
        self.emb.requires_grad_(requires_grad=False)
        self.rnn = nn.GRU(emb_dim,dim,batch_first=True)
        self.out = nn.Linear(dim,n_classes) 
        self.pool = AttentionPool(dim)
        
    def forward(self,x,test=False):
        mask = x == 0
        x = self.emb(x)
        rnn,_ = self.rnn(x) 
        x = self.pool(rnn,mask=mask)
        
        if test:
            return rnn
        
        return self.out(x)


In [9]:

import numpy as np

tok = WordTokenizer(10000)

def prepare_data(train_data,test_data,val_data,tok,elmo=False):

    xtrain,ytrain = train_data
    xtest,ytest = test_data
    xval,yval = val_data
    
    if elmo:

        elmo_train = [tree_bank.tokenize(x) for x in xtrain]
        elmo_test = [tree_bank.tokenize(x) for x in xtest]
        elmo_val = [tree_bank.tokenize(x) for x in xval]

        elmo_vocab_file = "vocab-2016-09-10.txt"
        batcher = Batcher(elmo_vocab_file,20,)
        elmo_train = batcher.batch_sentences(elmo_train)
        elmo_test = batcher.batch_sentences(elmo_test)
        elmo_val = batcher.batch_sentences(elmo_val)

    tok.fit(xtrain)
    xtrain = tok.encode_batch(xtrain)
    xtest = tok.encode_batch(xtest)
    xval = tok.encode_batch(xval)

    xtrain = np.array(xtrain)
    xtest = np.array(xtest)
    xval = np.array(xval)

    ytest = ytest.astype(np.int32)
    ytrain = ytrain.astype(np.int32)
    yval = yval.astype(np.int32)

    vocab = len(tok.word2id)
    
    data = {}
    
    if elmo:
        
        data['train'] = (xtrain,elmo_train,ytrain)
        data['test'] = (xtest,elmo_test,ytest)
        data['val'] = (xval,elmo_val,yval)
    else:
        data['train'] = (xtrain,ytrain)
        data['test'] = (xtest,ytest)
        data['val'] = (xval,yval)
    
    return vocab,data


In [10]:
from tqdm.auto import tqdm

def train_model(data,model,epochs=7):
    
    train,valid = data

    losses = {'train':[],'valid':[]}

    for e in range(1,epochs + 1):

        print(f"{e}/{epochs}")

        loss = 0
        model.train()

        for i,batch in enumerate(tqdm(train),1):

            lr = lr_scheduler[i]
            opt.param_groups[0]['lr'] = lr

            opt.zero_grad()

            x,y = batch
            if n_classes == 1:
                y = y.float().unsqueeze(1)
            elif n_classes > 1:
                y = y.long()
            pred = model(x)
            batch_loss = loss_fn(pred,y)
            batch_loss.backward()
            opt.step()

            loss += batch_loss.item()

        loss = round(loss / len(train),4)
        losses['train'].append(loss)

        print(f'Train Loss = {loss}')

        model.eval()
        loss = 0

        with torch.no_grad():

            for i,batch in enumerate(valid):

                x,y = batch
                if n_classes == 1:
                    y = y.float().unsqueeze(1)
                elif n_classes > 1:
                    y = y.long()
                pred = model(x)
                batch_loss = loss_fn(pred,y)
                loss += batch_loss.item()

        loss = round(loss / len(test),4)
        print(f"Val Loss = {loss}")


        if e == 1 or loss < min(losses['valid']):
            torch.save(model.state_dict(),'saved_weights.pt')
            print("--weights saved")

        losses['valid'].append(loss)

        print()

In [11]:
from sklearn.metrics import classification_report,f1_score,confusion_matrix,recall_score,accuracy_score
import seaborn as sb


def run_metrics(model,data,n_classes):
    
    test,ytest = data

    if n_classes == 1:

        preds = []

        for x,_ in test:
            pred = model(x)
            pred = torch.sigmoid(pred)
            pred = torch.round(pred).reshape(-1).detach().cpu().numpy()
            preds.extend(pred)

        print("f1 score :",round(f1_score(ytest,preds),4))
        print("recall score :",round(recall_score(ytest,preds),4))
        print("accuracy score :",round(accuracy_score(ytest,preds),4))

    elif n_classes > 1:

        preds = []

        for x,_ in test:
            pred = model(x)
            pred = torch.softmax(pred,-1)
            pred = torch.argmax(pred,1).cpu().numpy()
            preds.extend(pred)
        print("f1 score :",round(f1_score(ytest,preds,average='macro'),4))
        print("recall score :",round(recall_score(ytest,preds,average='macro'),4))
        print("accuracy score :",round(accuracy_score(ytest,preds),4))

In [12]:
tree_bank = TreebankWordTokenizer()

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("holdback.csv")
df = pd.concat([df_train,df_test]).reset_index(drop=True)

X = df.samples.apply(lambda x: preprocess_text(x))
Y = df.targets.values

xtrain,xtest,ytrain,ytest = train_test_split(X,Y,train_size=0.7,random_state=0)
xtest,xval,ytest,yval = train_test_split(xtest,ytest,train_size=0.66,random_state=0)

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [13]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok)

xtrain,ytrain = data['train']
xtest,ytest = data['test']
xval,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1
    
glove_emb = initialize_embeddings(vocab)

In [15]:
reset_random_seeds()

model = BaseClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model

BaseClassifier(
  (emb): Embedding(2426, 300)
  (rnn): GRU(300, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=3, bias=True)
  (pool): AttentionPool(
    (w): Linear(in_features=128, out_features=1, bias=False)
  )
)

In [16]:
from torch.utils.data import TensorDataset, DataLoader


BATCH = 4

def create_tensor_data(x,y,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    data = TensorDataset(x_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,BATCH)
test = create_tensor_data(xtest,ytest,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [17]:
train_model((train,valid),model)

1/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 1.0325
Val Loss = 0.5331
--weights saved

2/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 0.9505
Val Loss = 0.5218
--weights saved

3/7


...

Train Loss = 0.7861
Val Loss = 0.5269

6/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 0.7201
Val Loss = 0.5557

7/7


  0%|          | 0/175 [00:00<?, ?it/s]

Train Loss = 0.6719
Val Loss = 0.593



In [18]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [20]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.614
recall score : 0.614
accuracy score : 0.606


In [21]:
tree_bank = TreebankWordTokenizer()

df = pd.read_csv("/mnt/38DEEB97DEEB4C26/dataset/train.csv")

X = df.text.apply(lambda x: preprocess_text(x)).values
Y = df.target.values

xtrain,xtest,ytrain,ytest = train_test_split(X,Y,train_size=0.7,random_state=0)
xtest,xval,ytest,yval = train_test_split(xtest,ytest,train_size=0.66,random_state=0)

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [22]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok)

xtrain,ytrain = data['train']
xtest,ytest = data['test']
xval,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [24]:
reset_random_seeds()

glove_emb = initialize_embeddings(vocab)

model = BaseClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model

BaseClassifier(
  (emb): Embedding(10000, 300)
  (rnn): GRU(300, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=1, bias=True)
  (pool): AttentionPool(
    (w): Linear(in_features=128, out_features=1, bias=False)
  )
)

In [25]:
from torch.utils.data import TensorDataset, DataLoader


BATCH = 8

def create_tensor_data(x,y,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    data = TensorDataset(x_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,BATCH)
test = create_tensor_data(xtest,ytest,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [26]:
train_model((train,valid),model)

1/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.4739
Val Loss = 0.2283
--weights saved

2/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.4215
Val Loss = 0.2288

3/7


...

Train Loss = 0.3658
Val Loss = 0.2272

6/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.3501
Val Loss = 0.2334

7/7


  0%|          | 0/667 [00:00<?, ?it/s]

Train Loss = 0.3236
Val Loss = 0.2392



In [27]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [29]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.7699
recall score : 0.7456
accuracy score : 0.816


In [30]:
from datasets import load_dataset

tree_bank = TreebankWordTokenizer()

data = load_dataset("tweet_eval", "sentiment")
xtrain,ytrain = pd.DataFrame(data['train']).values.T
xtrain = [preprocess_text(x) for x in xtrain]


xtest,ytest = pd.DataFrame(data['test']).values.T
xtest = [preprocess_text(x) for x in xtest]

xval,yval = pd.DataFrame(data["validation"]).values.T
xval = [preprocess_text(x) for x in xval]

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [31]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok)

xtrain,ytrain = data['train']
xtest,ytest = data['test']
xval,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [33]:
reset_random_seeds()

glove_emb = initialize_embeddings(vocab)

model = BaseClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model

BaseClassifier(
  (emb): Embedding(10000, 300)
  (rnn): GRU(300, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=3, bias=True)
  (pool): AttentionPool(
    (w): Linear(in_features=128, out_features=1, bias=False)
  )
)

In [34]:
from torch.utils.data import TensorDataset, DataLoader


BATCH = 32

def create_tensor_data(x,y,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    data = TensorDataset(x_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,BATCH)
test = create_tensor_data(xtest,ytest,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [35]:
train_model((train,valid),model)

1/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.7876
Val Loss = 0.1277
--weights saved

2/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.7325
Val Loss = 0.1236
--weights saved

3/7


...

Train Loss = 0.6593
Val Loss = 0.1195
--weights saved

6/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.6384
Val Loss = 0.1191
--weights saved

7/7


  0%|          | 0/1426 [00:00<?, ?it/s]

Train Loss = 0.6169
Val Loss = 0.1199



In [36]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [38]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.6217
recall score : 0.6298
accuracy score : 0.636


In [39]:
tree_bank = TreebankWordTokenizer()

df_train = pd.read_csv("trec50_train.csv")
df_test = pd.read_csv("trec50_test.csv")

X = df_train.text.apply(lambda x: preprocess_text(x)).values
Y = df_train['label-fine'].values

xtrain,xval,ytrain,yval = train_test_split(X,Y,train_size=0.9,random_state=0)
xtest = df_test.text.apply(lambda x: preprocess_text(x)).values
ytest = df_test['label-fine'].values


In [40]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok)

xtrain,ytrain = data['train']
xtest,ytest = data['test']
xval,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [42]:
reset_random_seeds()

glove_emb = initialize_embeddings(vocab)

model = BaseClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model

BaseClassifier(
  (emb): Embedding(7957, 300)
  (rnn): GRU(300, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=47, bias=True)
  (pool): AttentionPool(
    (w): Linear(in_features=128, out_features=1, bias=False)
  )
)

In [43]:
BATCH = 8

def create_tensor_data(x,y,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    data = TensorDataset(x_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,BATCH)
test = create_tensor_data(xtest,ytest,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [44]:
train_model((train,valid),model)

1/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 2.4972
Val Loss = 2.3342
--weights saved

2/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 1.6618
Val Loss = 1.6906
--weights saved

3/7


...

Train Loss = 0.7585
Val Loss = 1.0704
--weights saved

6/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 0.6188
Val Loss = 1.0285
--weights saved

7/7


  0%|          | 0/614 [00:00<?, ?it/s]

Train Loss = 0.5001
Val Loss = 1.0132
--weights saved



In [45]:
state_dict = torch.load("saved_weights.pt", map_location='cuda')
model.load_state_dict(state_dict)

<All keys matched successfully>

In [47]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.573
recall score : 0.584
accuracy score : 0.79


In [48]:
df = pd.read_csv("/mnt/38DEEB97DEEB4C26/dataset/IMDB Dataset.csv")

X = df.review.apply(lambda x: preprocess_text(x)).values
df.sentiment = df.sentiment.map({'positive':1,'negative':0})
Y = df.sentiment.values

xtrain,xtest,ytrain,ytest = train_test_split(X,Y,train_size=0.7,random_state=0)
xtest,xval,ytest,yval = train_test_split(xtest,ytest,train_size=0.66,random_state=0)

maxlen = min(128-2,max([len(x.split()) for x in xtrain]))

tree_bank = TreebankWordTokenizer()

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtrain]
xtrain = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xtest]
xtest = [' '.join(x) for x in tok_x]

tok_x = [tree_bank.tokenize(x)[:maxlen] for x in xval]
xval = [' '.join(x) for x in tok_x]

In [49]:
vocab,data = prepare_data((xtrain,ytrain),(xtest,ytest),(xval,yval),tok)

xtrain,ytrain = data['train']
xtest,ytest = data['test']
xval,yval = data['val']

n_classes = len(np.unique(ytrain))

if n_classes == 2:
    n_classes = n_classes - 1

In [51]:
reset_random_seeds()

glove_emb = initialize_embeddings(vocab)

model = BaseClassifier(vocab,glove_emb)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

model

BaseClassifier(
  (emb): Embedding(10000, 300)
  (rnn): GRU(300, 128, batch_first=True)
  (out): Linear(in_features=128, out_features=1, bias=True)
  (pool): AttentionPool(
    (w): Linear(in_features=128, out_features=1, bias=False)
  )
)

In [52]:
from torch.utils.data import TensorDataset, DataLoader


BATCH = 32

def create_tensor_data(x,y,batch,shuffle=True):
    x_ = torch.tensor(x,dtype=torch.int32,device=device)
    y_ = torch.tensor(y,dtype=torch.int32,device=device)
    data = TensorDataset(x_,y_)
    data = DataLoader(data,batch_size=batch,shuffle=shuffle)
    return data


train = create_tensor_data(xtrain,ytrain,BATCH)
test = create_tensor_data(xtest,ytest,BATCH * 2,shuffle=False)
valid = create_tensor_data(xval,yval,BATCH * 2,shuffle=False)

opt = torch.optim.Adam(model.parameters())

if n_classes > 1:
    loss_fn = nn.CrossEntropyLoss()

else:
    loss_fn = nn.BCEWithLogitsLoss()

lr_scheduler = np.linspace(1e-3,1e-4,len(train)+1)

In [53]:
train_model((train,valid),model)

1/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.449
Val Loss = 0.2118
--weights saved

2/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3929
Val Loss = 0.1988
--weights saved

3/7


...

Train Loss = 0.3407
Val Loss = 0.192
--weights saved

6/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3254
Val Loss = 0.1884
--weights saved

7/7


  0%|          | 0/1094 [00:00<?, ?it/s]

Train Loss = 0.3093
Val Loss = 0.1875
--weights saved



In [55]:
run_metrics(model,(test,ytest),n_classes)

f1 score : 0.85
recall score : 0.8467
accuracy score : 0.8497
